### Assignment 4


Clustering is a quite demanding task. Keep in mind that different operations require different data
formats. In this part, we use three testfiles (files "cluster1a.arff”, “cluster2a.arff”,”cluster3a.arff"),
which can be found in the Moodle assignment folder. The files “cluster1a.arff” and “cluster2a.arff”
contain six attributes and the file “cluster3a.arff” contains 30 attributes. The file “cluster1a.arff”
contains 600 samples, the file “cluster2a.arff” contains 1000 samples, and the file “cluster3a.arff”
contains 1600 samples. Load the files and try to cluster the data.
What to report?
Answer the following questions and report the answers as well as the required Weka outputs in your
exercise report.
Questions
How many cluster you are able to find for each test case? (three files)
Where were the cluster centres located?
What was the best clustering approach to each test file? (describe it)
Was there any problems during the clustering.

a) Clustering and its outputs:

In [15]:
import pandas as pd
from scipy.io import arff
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

files = ["cluster1a.arff", "cluster2a.arff", "cluster3a.arff"]

for fname in files:
    data, _ = arff.loadarff(fname)
    df = pd.DataFrame(data).astype(float)
    X = StandardScaler().fit_transform(df)

    best_s, best_k, best_km = -1, None, None
    for k in range(2, 11):
        km = KMeans(k, n_init=10, random_state=0).fit(X)
        s = silhouette_score(X, km.labels_)
        if s > best_s:
            best_s, best_k, best_km = s, k, km

    hier = AgglomerativeClustering(n_clusters=best_k, linkage="ward").fit(X)

    em = GaussianMixture(n_components=best_k, random_state=0).fit(X)
    em_labels = em.predict(X)

    print(f"\n=== Results for {fname} ===")
    print(f"Instances: {df.shape[0]}")
    print(f"Attributes: {df.shape[1]}")
    print(f"Clusters used (KMeans/Hierarchical/EM): {best_k} / {best_k} / {best_k}")

    print("\n--- KMeans ---")
    print("Cluster centers:")
    km_centers = pd.DataFrame(best_km.cluster_centers_, columns=df.columns).round(2)
    km_centers.insert(0, "Cluster", range(best_k))
    print(km_centers.to_string(index=False))
    km_sizes = pd.Series(best_km.labels_).value_counts().sort_index().tolist()
    print(f"Cluster Sizes: {km_sizes}")

    print("\n--- Hierarchical ---")
    print("Cluster centers:")
    hier_centers = pd.DataFrame(
        [X[hier.labels_ == i].mean(0) for i in range(best_k)],
        columns=df.columns
    ).round(2)
    hier_centers.insert(0, "Cluster", range(best_k))
    print(hier_centers.to_string(index=False))
    hier_sizes = pd.Series(hier.labels_).value_counts().sort_index().tolist()
    print(f"Cluster Sizes: {hier_sizes}")

    print("\n--- EM Clustering ---")
    print("Cluster centers:")
    em_centers = pd.DataFrame(em.means_, columns=df.columns).round(2)
    em_centers.insert(0, "Cluster", range(best_k))
    print(em_centers.to_string(index=False))
    em_sizes = pd.Series(em_labels).value_counts().sort_index().tolist()
    print(f"Cluster Sizes: {em_sizes}")

    sil_kmeans = silhouette_score(X, best_km.labels_)
    sil_hier = silhouette_score(X, hier.labels_)
    sil_em = silhouette_score(X, em_labels)
    print(f"\nSilhouette score (KMeans): {sil_kmeans:.3f}")
    print(f"Silhouette score (Hierarchical): {sil_hier:.3f}")
    print(f"Silhouette score (EM): {sil_em:.3f}")



=== Results for cluster1a.arff ===
Instances: 600
Attributes: 6
Clusters used (KMeans/Hierarchical/EM): 3 / 3 / 3

--- KMeans ---
Cluster centers:
 Cluster    f1    f2   f3    f4    f5    f6
       0  0.00  0.00 -0.0 -0.00  0.00 -0.00
       1  1.18  1.19  1.2  1.19  1.19  1.19
       2 -1.18 -1.19 -1.2 -1.19 -1.19 -1.19
Cluster Sizes: [200, 200, 200]

--- Hierarchical ---
Cluster centers:
 Cluster    f1    f2   f3    f4    f5    f6
       0  1.18  1.19  1.2  1.19  1.19  1.19
       1 -1.18 -1.19 -1.2 -1.19 -1.19 -1.19
       2 -0.00  0.00 -0.0 -0.00  0.00 -0.00
Cluster Sizes: [200, 200, 200]

--- EM Clustering ---
Cluster centers:
 Cluster    f1    f2   f3    f4    f5    f6
       0  0.00  0.00 -0.0 -0.00  0.00 -0.00
       1  1.18  1.19  1.2  1.19  1.19  1.19
       2 -1.18 -1.19 -1.2 -1.19 -1.19 -1.19
Cluster Sizes: [200, 200, 200]

Silhouette score (KMeans): 0.729
Silhouette score (Hierarchical): 0.729
Silhouette score (EM): 0.729

=== Results for cluster2a.arff ===
Instances: 100

For the clustering tasks, I used KMeans, Hierarchical clustering (Ward method), and EM clustering. The cluster sizes, cluster centers, and other details are shown for each file above.

Overall, the silhouette scores for all methods and all datasets were similar and for KMeans and Hierarchical were identical. However, as the number of dimensions increased, EM showed slightly worse results. Because of it, KMeans or Hierarchical would be a better choice for these datasets. Looking deeper, I would prefer KMeans since it would probably be faster.

During the clustering, there were no major problems. Although I downloaded Weka to compare results, the EM cluster centers slightly differed from those produced by Weka’s EM, since the number of clusters was manually fixed here for more consistent comparison.